In [2]:
import numpy, pandas, mlxtend

1.26.4
2.2.2
0.23.1


In [8]:
import pandas as pd
from mlxtend.frequent_patterns import apriori, association_rules
# LOAD DATA

df = pd.read_csv("Airline_ETL_final.csv")

df_mine = df[[
    "Age_Group",
    "Continents",
    "is_weekend",
    "Flight Status"
]].astype(str)


df_encoded = pd.get_dummies(df_mine)

frequent = apriori(
    df_encoded,
    min_support=0.005,   # balanced threshold
    use_colnames=True
)

rules = association_rules(
    frequent,
    metric="confidence",
    min_threshold=0.3
)


rules = rules[
    rules['consequents'].apply(
        lambda x: len(x) == 1 and any("Flight Status" in item for item in x)
    )
]

rules = rules[
    rules['antecedents'].apply(lambda x: len(x) <= 2)
]

rules = rules.sort_values(
    by=["lift", "confidence"],
    ascending=False
)

top_rules = rules.head(5).copy()


# ===============================
def format_rule(x):
    return ', '.join([item.replace("_", " = ") for item in x])

top_rules["LHS"] = top_rules["antecedents"].apply(format_rule)
top_rules["RHS"] = top_rules["consequents"].apply(format_rule)

# ===============================
# FINAL OUTPUT TABLE
# ===============================
final_output = top_rules[[
    "LHS",
    "RHS",
    "support",
    "confidence",
    "lift"
]]

print("\nFinal Association Rules:\n")
print(final_output)


Final Association Rules:

                                                 LHS  \
242         Continents = Europe, Age = Group = Young   
327     is = weekend = 1, Continents = South America   
135  Continents = South America, Age = Group = Adult   
296            Continents = Europe, is = weekend = 1   
320           Continents = Oceania, is = weekend = 1   

                           RHS   support  confidence      lift  
242    Flight Status = Delayed  0.008264    0.349336  1.049348  
327  Flight Status = Cancelled  0.010860    0.348860  1.044388  
135  Flight Status = Cancelled  0.017644    0.346752  1.038076  
296    Flight Status = On Time  0.012148    0.343956  1.032717  
320    Flight Status = On Time  0.013882    0.343022  1.029911  
